# TAB 68A: exp68a_norml1 — EXP-68A (1 biến duy nhất so với exp63) - Colab
# Mục đích: kiểm chứng riêng H20a/FM22 trước khi đổi kiến trúc. Nếu 68a không chết death spiral
# thì "normalized L1" đủ; nếu 68a vẫn sụp thì lỗi nằm ở chỗ khác, không phải ở loss.
# Delta so với exp63 (test LP 0.6435 / maha 0.7134) CHỈ có 1 thứ: --pred_loss_type normalized_l1
# (z-score từng probe theo chiều feature):
#   - bất biến tuyệt đối với scale biểu diễn => xoá đường hạ loss bằng co rút biên độ (death
#     spiral của raw L1). Đo trên real data: bất biến tại x0.1 .. x100 (0.180713 ở mọi scale).
#   - vẫn phạt nặng dự đoán mù vị trí: trên feature thật, đoán token trung bình scan = 0.855 còn
#     đoán vị trí ngẫu nhiên = 0.944 => loss buộc phải dùng vị trí bị mask.
# Giữ nguyên 100% phần còn lại của exp63: 2-subspace, query_projection, scattering VICReg,
# inv 0.2 @ epoch 11 ramp 20, xcov 0.05, var_gamma 0.5, inv_predictor.
# LƯU Ý: --cov_weight KHÔNG tác dụng ở nhánh disentangled (nhánh dùng --scattering_cov_weight,
# exp63 đã = 0.2). Vì vậy 68a đúng nghĩa 1 biến, không phải 2.
# Bar: R_eff >= 15D ở epoch 20-30; pred_loss KHÔNG được xuống < 0.35 khi test LP < 0.62.

# VẬN HÀNH (đã sửa): `runs/` được symlink sang Drive + assert kiểm chứng; cell cuối copy
# dự phòng nếu symlink hỏng. Preflight kiểm tra `train.py` có đủ flag trước khi train
# (lỗi "unrecognized arguments" của 68B là do clone Colab thiếu code mới).

In [8]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [9]:
import os, glob
DATA = '/content/drive/MyDrive/PECT_Dataset/'
DRIVE_RUNS = '/content/drive/MyDrive/PECT_Dataset/runs/'
os.makedirs(DRIVE_RUNS, exist_ok=True)
# Verify: TDMS + GT masks phai thay tu Drive
tdms = [f for f in glob.glob(os.path.join(DATA, '**', '*.tdms'), recursive=True) if not f.endswith('_index.tdms')]
print('TDMS:', len(tdms))
for p in [os.path.join(DATA, 'ground_truth/corrosion/corrosion_gt_mask.npy'),
          os.path.join(DATA, 'ground_truth/rivet_v1/rivet_v1_gt_mask.npy'),
          os.path.join(DATA, 'ground_truth/rivet_v2/rivet_v2_gt_mask.npy')]:
    print(('OK  ' if os.path.exists(p) else 'MISS') + ' ' + p)

TDMS: 81
OK   /content/drive/MyDrive/PECT_Dataset/ground_truth/corrosion/corrosion_gt_mask.npy
OK   /content/drive/MyDrive/PECT_Dataset/ground_truth/rivet_v1/rivet_v1_gt_mask.npy
OK   /content/drive/MyDrive/PECT_Dataset/ground_truth/rivet_v2/rivet_v2_gt_mask.npy


In [10]:
!test -d GraduationProject && (cd GraduationProject && git fetch origin && git checkout Define_PECT_JEPA && git reset --hard origin/Define_PECT_JEPA) || git clone -b Define_PECT_JEPA https://github.com/TranAnh35/GraduationProject.git
!cd GraduationProject && git log --oneline -3

remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 6 (delta 5), reused 6 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 521 bytes | 130.00 KiB/s, done.
From https://github.com/TranAnh35/GraduationProject
   08f7743..1392b95  Define_PECT_JEPA -> origin/Define_PECT_JEPA
Already on 'Define_PECT_JEPA'
Your branch is behind 'origin/Define_PECT_JEPA' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)
HEAD is now at 1392b95 fix(eval): fix NameError cached_train_latents typo in train readout fitting fallback
1392b95 (HEAD -> Define_PECT_JEPA, origin/Define_PECT_JEPA) fix(eval): fix NameError cached_train_latents typo in train readout fitting fallback
08f7743 feat(eval): add --skip_domain_adaptation flag and update knowledge graph
0bf6243 test(cli): cover the train.py argparse surface that the EXP-68B crash fell through


In [11]:
# [FIX] sklearn version co sample_weight trong MLPClassifier.fit -> ep dung 1.7.2 (giong env lab)
!pip install -q -U "scikit-learn==1.7.2" nptdms scipy tqdm 2>&1 | tail -2
!cd GraduationProject && python -c "import torch, sklearn, scipy, nptdms; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(), '| sklearn:', sklearn.__version__)"

torch 2.11.0+cu130 | cuda: True | sklearn: 1.7.2


In [12]:
# ==============================================================================
# PREFLIGHT: code tren Colab PHAI la ban moi nhat. Neu thieu flag, train.py se chet voi
# "error: unrecognized arguments" (da xay ra voi EXP-68B vi clone cu chua co flag moi).
# ==============================================================================
import subprocess

REQUIRED_FLAGS = ['--spectral_condition_token', '--spectral_phase_encoding',
                  '--ref_patch_mode', '--unified_latent', '--pred_loss_type',
                  '--self_focus_gamma', '--num_target_rays']

TRAIN_PY = 'GraduationProject/src/PECT_JEPA/cross_condition/train.py'
src = open(TRAIN_PY, encoding='utf-8').read()
missing = [f for f in REQUIRED_FLAGS if f not in src]

head = subprocess.run(['git', '-C', 'GraduationProject', 'rev-parse', '--short', 'HEAD'],
                      capture_output=True, text=True).stdout.strip()
branch = subprocess.run(['git', '-C', 'GraduationProject', 'rev-parse', '--abbrev-ref', 'HEAD'],
                        capture_output=True, text=True).stdout.strip()
print(f'[preflight] branch={branch} HEAD={head}')
print(f'[preflight] flags: {"OK" if not missing else "THIEU " + str(missing)}')
if missing:
    raise SystemExit(
        'train.py tren Colab thieu flag: ' + ', '.join(missing) +
        '\n-> Chay lai cell git o tren (git pull origin Define_PECT_JEPA) roi chay lai cell nay. '
        'KHONG chay cell Train khi con thieu flag.')

[preflight] branch=Define_PECT_JEPA HEAD=1392b95
[preflight] flags: OK


In [13]:
# ==============================================================================
# Gan experiments/cross_condition/runs -> Google Drive
# Checkpoint, metrics_epoch.csv, run.log ghi truc tiep len Drive (khong mat khi Colab ngat).
# Cell tu kiem chung: assert se dung notebook neu symlink khong tro ve Drive.
# ==============================================================================
import os, shutil, subprocess

RUNS_LINK = 'GraduationProject/experiments/cross_condition/runs'
os.makedirs(DRIVE_RUNS, exist_ok=True)
os.makedirs(os.path.dirname(RUNS_LINK), exist_ok=True)

if os.path.islink(RUNS_LINK):
    print('ALREADY_LINKED ->', os.path.realpath(RUNS_LINK))
else:
    if os.path.isdir(RUNS_LINK):
        for name in os.listdir(RUNS_LINK):
            s, d = os.path.join(RUNS_LINK, name), os.path.join(DRIVE_RUNS, name)
            if not os.path.exists(d):
                shutil.copytree(s, d)
                print('  migrated', name)
        shutil.rmtree(RUNS_LINK)
    os.symlink(DRIVE_RUNS, RUNS_LINK)
    print('LINKED', RUNS_LINK, '->', DRIVE_RUNS)

assert os.path.realpath(RUNS_LINK) == os.path.realpath(DRIVE_RUNS), \
    f'Symlink KHONG tro ve Drive: {os.path.realpath(RUNS_LINK)}'
print(subprocess.run(['ls', '-la', os.path.dirname(RUNS_LINK)],
                     capture_output=True, text=True).stdout)

ALREADY_LINKED -> /content/drive/MyDrive/PECT_Dataset/runs
total 8
drwxr-xr-x 2 root root 4096 Oct 22 02:58 .
drwxr-xr-x 4 root root 4096 Oct 22 02:58 ..
lrwxrwxrwx 1 root root   41 Oct 22 02:58 runs -> /content/drive/MyDrive/PECT_Dataset/runs/



In [14]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

# ===== EXP-68A CONFIG: exp63 + probe-wise normalized L1 (single-variable test) =====
EXP_NAME = 'exp68a_norml1'

BASE_FLAGS = '--star_radii 2 4 7 --tokenizer_type energy_adaptive --embed_dim 64 --latent_disentanglement true --carrier_flat_weight 0.0 --scattering_var_weight 0.5 --scattering_cov_weight 0.2 --var_weight 0.5 --var_gamma 0.5 --scan_centered_vicreg true --continuous_pos_embed true --contiguous_masking true --relative_energy_anchor true --bi_directional_relative_energy true --patch_polarity_align true --use_condition_token true --condition_modulation query_projection --inv_weight 0.2 --inv_start_epoch 11 --inv_ramp_epochs 20 --inv_cosine true --inv_predictor true --scan_centered_inv true --xcov_weight 0.05 --xcov_start_epoch 11 --xcov_ramp_epochs 20 --zc_cond_weight 0.1 --diverge_patience 3 --device cuda'
EXTRA = '--pred_loss_type normalized_l1'

os.environ.update({'EXP_NAME': EXP_NAME, 'BASE_FLAGS': BASE_FLAGS, 'EXTRA': EXTRA, 'DATA': DATA, 'DRIVE_RUNS': DRIVE_RUNS})
print('EXPERIMENT:', EXP_NAME)
print('FLAGS:', BASE_FLAGS + ' ' + EXTRA)

EXPERIMENT: exp68a_norml1
FLAGS: --star_radii 2 4 7 --tokenizer_type energy_adaptive --embed_dim 64 --latent_disentanglement true --carrier_flat_weight 0.0 --scattering_var_weight 0.5 --scattering_cov_weight 0.2 --var_weight 0.5 --var_gamma 0.5 --scan_centered_vicreg true --continuous_pos_embed true --contiguous_masking true --relative_energy_anchor true --bi_directional_relative_energy true --patch_polarity_align true --use_condition_token true --condition_modulation query_projection --inv_weight 0.2 --inv_start_epoch 11 --inv_ramp_epochs 20 --inv_cosine true --inv_predictor true --scan_centered_inv true --xcov_weight 0.05 --xcov_start_epoch 11 --xcov_ramp_epochs 20 --zc_cond_weight 0.1 --diverge_patience 3 --device cuda --pred_loss_type normalized_l1


In [ ]:
# Cell Train (Fresh Run): exp68a_norml1 — normalized L1
!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.train --exp_name $EXP_NAME --add_timestamp true $BASE_FLAGS $EXTRA --data_dir "$DATA" --epochs 100 --eval_val_every 1

In [15]:
# ==============================================================================
# Xac nhan run folder DA NAM TREN DRIVE. Neu run dir khong nam trong DRIVE_RUNS
# (symlink hong / bi thay bang folder that) thi copy sang Drive ngay tai day.
# ==============================================================================
import glob, os, shutil

cands = sorted(glob.glob(os.path.join(DRIVE_RUNS, f'{EXP_NAME}_*')), key=os.path.getmtime)
run_dir = os.environ.get('LAST_RUN_DIR') or (cands[-1] if cands else None)
if not run_dir or not os.path.isdir(run_dir):
    raise FileNotFoundError(f'Khong thay run {EXP_NAME} tren Drive: {DRIVE_RUNS}')

real_run = os.path.realpath(run_dir)
real_drive = os.path.realpath(DRIVE_RUNS)
if not real_run.startswith(real_drive):
    dst = os.path.join(DRIVE_RUNS, os.path.basename(real_run))
    shutil.copytree(real_run, dst, dirs_exist_ok=True)
    print('[copy] run dir KHONG nam tren Drive -> da copy sang', dst)
    run_dir = dst

print('[*] run dir tren Drive:', run_dir)
for f in ['config.json', 'run.log', 'metrics_epoch.csv',
          'checkpoints/best_cross_condition_jepa.pt',
          'checkpoints/latest_cross_condition_jepa.pt']:
    p = os.path.join(run_dir, f)
    print(('  OK   ' if os.path.exists(p) else '  MISS ') + f)
print('[*] entries:', sorted(os.listdir(run_dir)))

[*] run dir tren Drive: /content/drive/MyDrive/PECT_Dataset/runs/exp68a_norml1_20261021_141121
  OK   config.json
  OK   run.log
  OK   metrics_epoch.csv
  OK   checkpoints/best_cross_condition_jepa.pt
  OK   checkpoints/latest_cross_condition_jepa.pt
[*] entries: ['checkpoints', 'config.json', 'config_resume.json', 'evaluation_results', 'git_info.json', 'metrics_epoch.csv', 'results', 'run.log', 'visualizations']


In [ ]:
# ==============================================================================
# Resume training (CHI chay khi session truoc da dung han).
# Bat buoc truyen lai $BASE_FLAGS $EXTRA: train.py dung checkpoint de suy ra run_dir,
# nhung config/architecture van duoc dung tu flag -> thieu flag = sai kien truc, load_state_dict fail.
# ==============================================================================
import glob, os

SPECIFIC_RUN = ''  # de trong = run moi nhat cua EXP_NAME

cands = sorted(glob.glob(os.path.join(DRIVE_RUNS, f'{EXP_NAME}_*')), key=os.path.getmtime)
if not cands:
    raise FileNotFoundError(f'Khong thay run {EXP_NAME} trong {DRIVE_RUNS}')
run_dir = os.path.join(DRIVE_RUNS, SPECIFIC_RUN) if SPECIFIC_RUN else cands[-1]

ck_latest = os.path.join(run_dir, 'checkpoints', 'latest_cross_condition_jepa.pt')
ck_best = os.path.join(run_dir, 'checkpoints', 'best_cross_condition_jepa.pt')
if not (os.path.exists(ck_latest) or os.path.exists(ck_best)):
    raise FileNotFoundError(f'Khong co checkpoint trong {os.path.join(run_dir, "checkpoints")}')
resume_ckpt = ck_latest if os.path.exists(ck_latest) else ck_best
print('[*] run_dir    :', run_dir)
print('[*] resume ckpt:', resume_ckpt, f'({os.path.getsize(resume_ckpt)/1024**2:.1f} MB)')
os.environ['RESUME_CKPT'] = resume_ckpt
os.environ['LAST_RUN_DIR'] = run_dir

!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.train --exp_name $EXP_NAME $BASE_FLAGS $EXTRA --data_dir "$DATA" --epochs 100 --eval_val_every 1 --resume_checkpoint "$RESUME_CKPT"

In [ ]:
# ==============================================================================
# Xac dinh run folder (moi nhat cua EXP_NAME) + checkpoint de eval/downstream
# ==============================================================================
import glob, os

SPECIFIC_RUN = ''  # de trong = run moi nhat

cands = sorted(glob.glob(os.path.join(DRIVE_RUNS, f'{EXP_NAME}_*')), key=os.path.getmtime)
if not cands:
    raise FileNotFoundError(f'Khong thay run {EXP_NAME} trong {DRIVE_RUNS}. Chay cell Train truoc.')
run_dir = os.path.join(DRIVE_RUNS, SPECIFIC_RUN) if SPECIFIC_RUN else cands[-1]
os.environ['LAST_RUN_DIR'] = run_dir

ck_best = os.path.join(run_dir, 'checkpoints', 'best_cross_condition_jepa.pt')
ck_latest = os.path.join(run_dir, 'checkpoints', 'latest_cross_condition_jepa.pt')
if not (os.path.exists(ck_best) or os.path.exists(ck_latest)):
    raise FileNotFoundError(f'Khong co checkpoint trong {os.path.join(run_dir, "checkpoints")}')
eval_ckpt = ck_best if os.path.exists(ck_best) else ck_latest
os.environ['EVAL_CKPT'] = eval_ckpt
print('[*] run_dir :', run_dir)
print('[*] ckpt    :', eval_ckpt, f'({os.path.getsize(eval_ckpt)/1024**2:.1f} MB)')

In [ ]:
# ==============================================================================
# (Tuy chon) Train downstream: Linear/MLP probes + Sizing (bo qua 7 model UDA voi --skip_uda true)
# ==============================================================================
!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.downstream.train_downstream --checkpoint "$EVAL_CKPT" --data_dir "$DATA" --device cuda --skip_uda true

In [ ]:
# ==============================================================================
# Full downstream evaluation (65 test scans: Linear Probe, Mahalanobis, Sizing, UDA neu co)
# + readout diagnostics
# ==============================================================================
!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.evaluate --checkpoint "$EVAL_CKPT" --data_dir "$DATA" --device cuda --skip_domain_adaptation true
!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.diagnose_readouts --checkpoint "$EVAL_CKPT" --data_dir "$DATA" --device cuda


PECT-JEPA Unified Downstream Benchmark Evaluation
Checkpoint:       /content/drive/MyDrive/PECT_Dataset/runs/exp68a_norml1_20261021_141121/checkpoints/best_cross_condition_jepa.pt
Output Directory: /content/drive/MyDrive/PECT_Dataset/runs/exp68a_norml1_20261021_141121/evaluation_results
Compute Device:   cuda
Target Splits:    ['train', 'val', 'test']

Discovered 81 total inspection files:
  - Train Split (Base Domain):     12 files
  - Val Split (Base Domain):       4 files
  - Test Split (Held-out OOD):     65 files

STEP 1: Downstream Probes & UDA Models Setup

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
  [WARNING] No pre-trained downstream models found!
  Checked candidate paths:
    - /content/drive/MyDrive/PECT_Dataset/runs/exp68a_norml1_20261021_141121/downstream_models (exists: False)
    - /content/drive/MyDrive/PECT_Dataset/runs/exp68a_norml1_20261021_141121/downstream_models (exists: False)
    - /content/drive/MyDrive/PECT_Dataset/runs/exp6


Exception ignored in: <generator object _iter_with_prefetch at 0x7df0008aac00>
Traceback (most recent call last):
  File "/content/GraduationProject/src/PECT_JEPA/cross_condition/evaluate.py", line 231, in _iter_with_prefetch
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
  File "/usr/lib/python3.13/concurrent/futures/_base.py", line 653, in __exit__
    self.shutdown(wait=True)
  File "/usr/lib/python3.13/concurrent/futures/thread.py", line 239, in shutdown
^C
